In [1]:
%reload_ext dotenv
%dotenv

import os
from pathlib import Path
from typing import List
from dataclasses import dataclass, field

# ―――― local helper dataclasses ――――
@dataclass
class EntityType:
    name: str
    description: str
    examples: List[str] = field(default_factory=list)

@dataclass
class RelationType:
    name: str
    description: str
    examples: List[str] = field(default_factory=list)
# ───────────────────────────────────

from graphrag_toolkit.lexical_graph import (
    LexicalGraphIndex,
    GraphRAGConfig,
    set_logging_config,
)
from graphrag_toolkit.lexical_graph.storage import GraphStoreFactory, VectorStoreFactory
from graphrag_toolkit.lexical_graph.indexing.load import FileBasedDocs
from graphrag_toolkit.lexical_graph.indexing.build import Checkpoint

# PDFReader lives in the extra “file” reader package
from llama_index.readers.file import PDFReader

set_logging_config("INFO")

In [ ]:
# # ---------------------------------------------------------------------------
# #  Custom entity / relation ontology
# # ---------------------------------------------------------------------------
# CUSTOM_ENTITY_TYPES = [
#     EntityType(
#         name="Company",
#         description="Companies and organisations involved in the semiconductor industry",
#         examples=["Intel", "TSMC", "NXP Semiconductors", "Taiwan Semiconductor"],
#     ),
#     EntityType(
#         name="Country",
#         description="Countries and regions involved in semiconductor manufacturing and trade",
#         examples=["Taiwan", "United States", "China", "South Korea"],
#     ),
#     EntityType(
#         name="Technology",
#         description="Semiconductor technologies, products, and components",
#         examples=["microchips", "semiconductors", "transistors", "advanced chips"],
#     ),
#     EntityType(
#         name="Policy",
#         description="Government policies, acts, and regulations",
#         examples=["CHIPS Act", "EU Chips Act", "export restrictions"],
#     ),
#     EntityType(
#         name="Resource",
#         description="Raw materials and resources used in semiconductor manufacturing",
#         examples=["neon", "palladium", "silicon"],
#     ),
#     EntityType(
#         name="MarketMetric",
#         description="Market shares, production capacities, and industry metrics",
#         examples=["65 percent market share", "90 percent advanced chips", "11.8 billion transistors"],
#     ),
# ]

# CUSTOM_RELATION_TYPES = [
#     RelationType(
#         name="produces",
#         description="Entity A manufactures Entity B",
#         examples=["Taiwan produces 65 percent of microchips", "TSMC produces advanced semiconductors"],
#     ),
#     RelationType(
#         name="regulates",
#         description="Entity A regulates or controls Entity B",
#         examples=["CHIPS Act regulates semiconductor manufacturing", "US restricts chip exports"],
#     ),
#     RelationType(
#         name="depends_on",
#         description="Entity A depends on Entity B",
#         examples=["chip production depends on neon", "military systems depend on semiconductors"],
#     ),
#     RelationType(
#         name="competes_with",
#         description="Entity A competes with Entity B",
#         examples=["US competes with China", "companies compete for semiconductor capacity"],
#     ),
#     RelationType(
#         name="supplies",
#         description="Entity A supplies Entity B",
#         examples=["Ukraine supplies neon", "Taiwan supplies advanced chips"],
#     ),
# ]

# # ---------------------------------------------------------------------------
# #  Configure the global GraphRAGConfig singleton in-place
# # ---------------------------------------------------------------------------
# GraphRAGConfig.entity_types      = CUSTOM_ENTITY_TYPES
# GraphRAGConfig.relation_types    = CUSTOM_RELATION_TYPES
# GraphRAGConfig.chunk_size        = 300      # smaller chunks for dense content
# GraphRAGConfig.chunk_overlap     = 100      # overlap to retain context
# GraphRAGConfig.extraction_prompt = """
# Analyse the following text about the semiconductor industry and extract:
#   1. Companies and organisations involved
#   2. Countries and their roles in semiconductor manufacturing
#   3. Technologies and products mentioned
#   4. Government policies and regulations
#   5. Raw materials and resources
#   6. Market metrics and statistics
#   7. Relationships between these entities (production, supply, regulation, competition, dependency)

# Focus on geopolitical and industry relationships that impact semiconductor supply-chains.
# """.strip()

# # ---------------------------------------------------------------------------
# #  Pipeline set-up (unchanged from original logic)
# # ---------------------------------------------------------------------------
# extracted_docs = FileBasedDocs(docs_directory="extracted")
# checkpoint     = Checkpoint("extraction-checkpoint")

# graph_store  = GraphStoreFactory.for_graph_store(os.environ["GRAPH_STORE"])
# vector_store = VectorStoreFactory.for_vector_store(os.environ["VECTOR_STORE"])

# graph_index = LexicalGraphIndex(          # uses the global GraphRAGConfig we just patched
#     graph_store=graph_store,
#     vector_store=vector_store,
    
# )



# # ---------------------------------------------------------------------------
# #  PDF ingest and extraction
# # ---------------------------------------------------------------------------
# pdf_dir = Path("data/pdfs")
# pdf_dir.mkdir(parents=True, exist_ok=True)

# pdf_files = list(pdf_dir.glob("*.pdf"))
# if not pdf_files:
#     print(f"No PDF files found in {pdf_dir}. "
#           "Add PDFs to data/pdfs/ and re-run this cell.")
# else:
#     print(f"Found {len(pdf_files)} PDF file(s):")
#     for pdf in pdf_files:
#         print(f"  • {pdf.name}")

#     print("\nLoading and extracting documents …")
#     # Process each PDF file individually and combine the results
#     docs = []
#     for pdf_file in pdf_files:
#         print(f"\nProcessing {pdf_file.name}:")
#         file_docs = PDFReader().load_data(pdf_file)
#         print(f"- Extracted {len(file_docs)} pages")
        
#         # Add source metadata to each page
#         for i, doc in enumerate(file_docs):
#             doc.metadata.update({
#                 "source": str(pdf_file),
#                 "page_number": i + 1,
#                 "total_pages": len(file_docs)
#             })
#             docs.append(doc)
        
#         print(f"- Added metadata to {len(file_docs)} pages")

#     print(f"\nTotal documents to process: {len(docs)}")
    
#     graph_index.extract(
#         docs,
#         handler=extracted_docs,
#         checkpoint=checkpoint,
#         show_progress=True,
#     )

#     collection_id = extracted_docs.collection_id
#     print("\nExtraction complete ✓")
#     print("collection_id:", collection_id)

In [6]:
# ---------------------------------------------------------------------------
#  Custom extraction / build configuration
# ---------------------------------------------------------------------------

# 1) Domain-specific prompt – add or edit wording as you wish
CUSTOM_TOPICS_PROMPT = """
You are analysing text about the semiconductor industry.  Extract ONLY:
• Entities of type  Company, Country, Technology, Policy, Resource, MarketMetric
• Relationships of type produces, regulates, depends_on, competes_with, supplies

Do not invent new entity classes or relationship names.

<preferredEntityClassifications>
Company
Country
Technology
Policy
Resource
MarketMetric
</preferredEntityClassifications>
""".strip()

# 2) List of preferred classifications (must match the six types above)
CUSTOM_CLASSIFICATIONS = ["Company", "Country", "Technology", "Policy", "Resource", "MarketMetric"]

# 3) Build an indexing configuration that uses the prompt & classes
from graphrag_toolkit.lexical_graph import (
    IndexingConfig,
    ExtractionConfig,
    BuildConfig,
)

from llama_index.core.node_parser import SentenceSplitter

indexing_config = IndexingConfig(
    # ①  CHUNKER – required
    chunking=[
        SentenceSplitter(
            chunk_size=300,       # ← hard-code instead of GraphRAGConfig.chunk_size
            chunk_overlap=100     # ← likewise
        )
    ],

    # ②  Extraction settings
    extraction=ExtractionConfig(
        preferred_entity_classifications=CUSTOM_CLASSIFICATIONS,
        extract_topics_prompt_template=CUSTOM_TOPICS_PROMPT,
        enable_proposition_extraction=True,
    ),

    # ③  Build settings
    build=BuildConfig(
        include_domain_labels=True
    ),
)

# ---------------------------------------------------------------------------
#  Pipeline set-up (uses the new indexing_config)
# ---------------------------------------------------------------------------
extracted_docs = FileBasedDocs(docs_directory="extracted")
checkpoint     = Checkpoint("extraction-checkpoint")

graph_store  = GraphStoreFactory.for_graph_store(os.environ["GRAPH_STORE"])
vector_store = VectorStoreFactory.for_vector_store(os.environ["VECTOR_STORE"])

graph_index = LexicalGraphIndex(                # ← now receives the config
    graph_store=graph_store,
    vector_store=vector_store,
    indexing_config=indexing_config
)

# ---------------------------------------------------------------------------
#  PDF ingest and extraction
# ---------------------------------------------------------------------------
pdf_dir = Path("data/pdfs")
pdf_dir.mkdir(parents=True, exist_ok=True)

pdf_files = list(pdf_dir.glob("*.pdf"))
if not pdf_files:
    print(f"No PDF files found in {pdf_dir}. "
          "Add PDFs to data/pdfs/ and re-run this cell.")
else:
    print(f"Found {len(pdf_files)} PDF file(s):")
    for pdf in pdf_files:
        print(f"  • {pdf.name}")

    print("\nLoading and extracting documents …")
    # Process each PDF file individually and combine the results
    docs = []
    for pdf_file in pdf_files:
        print(f"\nProcessing {pdf_file.name}:")
        file_docs = PDFReader().load_data(pdf_file)
        print(f"- Extracted {len(file_docs)} pages")
        
        # Add source metadata to each page
        for i, doc in enumerate(file_docs):
            doc.metadata.update({
                "source": str(pdf_file),
                "page_number": i + 1,
                "total_pages": len(file_docs)
            })
            docs.append(doc)
        
        print(f"- Added metadata to {len(file_docs)} pages")

    print(f"\nTotal documents to process: {len(docs)}")
    
    graph_index.extract(
        docs,
        handler=extracted_docs,
        checkpoint=checkpoint,
        show_progress=True,
    )

    collection_id = extracted_docs.collection_id
    print("\nExtraction complete ✓")
    print("collection_id:", collection_id)

Found 1 PDF file(s):
  • New-InfoFlash.Sample.pdf

Loading and extracting documents …

Processing New-InfoFlash.Sample.pdf:
- Extracted 4 pages
- Added metadata to 4 pages

Total documents to process: 4
2025-07-07 08:44:21:INFO:g.l.i.e.extraction_pipeline:Running extraction pipeline [batch_size: 4, num_workers: 2]


Extracting propositions [nodes: 4, num_workers: 4]: 100%|██████████| 4/4 [00:03<00:00,  1.10it/s]
Extracting propositions [nodes: 5, num_workers: 4]: 100%|██████████| 5/5 [00:04<00:00,  1.20it/s]
Extracting topics [nodes: 5, num_workers: 4]: 100%|██████████| 5/5 [00:05<00:00,  1.14s/it]


2025-07-07 08:44:35:INFO:g.l.i.b.build_pipeline:Running build pipeline [batch_size: 4, num_workers: 1, job_sizes: [20], batch_writes_enabled: True, batch_write_size: 25]

Extraction complete ✓
collection_id: 20250707-084420


In [7]:
%reload_ext dotenv
%dotenv

import os

from graphrag_toolkit.lexical_graph import LexicalGraphIndex, set_logging_config
from graphrag_toolkit.lexical_graph.storage import GraphStoreFactory
from graphrag_toolkit.lexical_graph.storage import VectorStoreFactory
from graphrag_toolkit.lexical_graph.indexing.load import FileBasedDocs
from graphrag_toolkit.lexical_graph.indexing.build import Checkpoint

set_logging_config('INFO')

# Initialize document storage with the collection ID from extraction
docs = FileBasedDocs(
    docs_directory='extracted',
    collection_id=collection_id
)
checkpoint = Checkpoint('build-checkpoint')

# Initialize stores
graph_store = GraphStoreFactory.for_graph_store(os.environ['GRAPH_STORE'])
vector_store = VectorStoreFactory.for_vector_store(os.environ['VECTOR_STORE'])

# Initialize graph index
graph_index = LexicalGraphIndex(
    graph_store, 
    vector_store
)

# Build the graph
print("Building knowledge graph...")
graph_index.build(docs, checkpoint=checkpoint, show_progress=True)

print('Build complete')


Building knowledge graph...
2025-07-07 08:45:33:INFO:g.l.i.b.build_pipeline:Running build pipeline [batch_size: 4, num_workers: 2, job_sizes: [9, 11], batch_writes_enabled: True, batch_write_size: 25]


Building graph [batch_writes_enabled: True, batch_write_size: 25]: 100%|██████████| 11/11 [00:00<00:00, 76895.57it/s]

Building vector index [batch_writes_enabled: True, batch_write_size: 25]: 100%|██████████| 11/11 [00:00<00:00, 334328.58it/s]
Building vector index [batch_writes_enabled: True, batch_write_size: 25]: 100%|██████████| 9/9 [00:00<00:00, 337042.29it/s]


Build complete
